In [ ]:
import torch
from transformers import (
    LlamaTokenizer,
    LlamaForCausalLM,
    GenerationConfig,
    BitsAndBytesConfig,
)
from peft import PeftModel

device = "cuda"
quantization_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.bfloat16,
)

model_name = "JG22/decapoda-research-llama-13b"
tokenizer = LlamaTokenizer.from_pretrained(model_name)

model = LlamaForCausalLM.from_pretrained(
    model_name,
    device_map="auto",
    quantization_config=quantization_config,
)
model = PeftModel.from_pretrained(model, "andreabac3/Fauno-Italian-LLM-13B")

model.eval()

In [ ]:
import re


def evaluate(
    prompt: str,
    question: str,
    gen_cfg: GenerationConfig,
    print_full: bool = False,
) -> str:
    model_input = prompt + f"\n[|Human|] {question}\n [|AI|] "
    inputs = tokenizer(model_input, return_tensors="pt")
    input_ids = inputs["input_ids"].to(device)

    generation_output = model.generate(
        input_ids=input_ids,
        generation_config=gen_cfg,
        return_dict_in_generate=True,
        output_scores=True,
        max_new_tokens=256,
    )
    decoded = tokenizer.decode(generation_output.sequences[0], skip_special_tokens=True)
    match = re.search(
        r"\[\|AI\|\](.*?)(?=\[\|?(Human|Umano|Uman)\|\]|$)", decoded, re.DOTALL
    )
    output = match.group(1).strip() if match else decoded

    if print_full:
        print(decoded)
        print("\n--- Full answer ---\n")

    return output


gen_cfg = GenerationConfig(
    temperature=1.0,
    top_p=0.95,
    num_beams=1,
    max_context_length_tokens=2048,
    do_sample=True,
)

prompt = "Sei un assistente AI chiamato Fauno. Il tuo scopo è di aiutare le persone a risolvere i loro problemi. Sei in grado di capire la lingua naturale e di rispondere alle domande dei tuoi clienti."
question = "Qual è il significato della vita?"

answer = evaluate(prompt, question, gen_cfg)
print(answer)
